In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv("de-gtfsde-TripUpdate-merged.csv")



C:\Users\nuece\AppData\Local\Temp\ipykernel_38276\3995123433.py:4: DtypeWarning: Columns (2: feed_id) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("de-gtfsde-TripUpdate-merged.csv")


In [15]:
stops = pd.read_csv("stops.txt")
print(stops.shape[0])
# Remove duplicate stop names from the stops dataframe
stops = stops.drop_duplicates(subset=['stop_name'])
print(stops)
print(stops.shape[0])



693585
                                         stop_name parent_station stop_id  \
0                      's-Heerenberg Gouden Handen            NaN    c05a   
2                         's-Heerenberg Molenpoort            NaN    9htv   
4                       's-Heerenberg Muziekschool            NaN    aozx   
7       (Planung) Brokdorf, Peuser (West), Mast 2r           cjps    bx2l   
8       (Planung) Brokdorf, Peuser (West), Mast 5l           cjps    6dqw   
...                                            ...            ...     ...   
693575                       Šibenik (Bus Station)            NaN    5a00   
693577                               Ždiar (Tatra)            NaN    7s29   
693579                        Žilina (bus station)            NaN    7o0b   
693581                                       Žireč            NaN     a4w   
693583                         Županja bus station            NaN    eg5i   

         stop_lat   stop_lon  location_type platform_code  
0       

In [16]:
stops = stops[['stop_name', 'stop_lat', 'stop_lon']]
print(stops)

                                         stop_name   stop_lat   stop_lon
0                      's-Heerenberg Gouden Handen  51.872250   6.247338
2                         's-Heerenberg Molenpoort  51.876320   6.247432
4                       's-Heerenberg Muziekschool  51.874790   6.254206
7       (Planung) Brokdorf, Peuser (West), Mast 2r  53.869694   9.342648
8       (Planung) Brokdorf, Peuser (West), Mast 5l  53.869675   9.342468
...                                            ...        ...        ...
693575                       Šibenik (Bus Station)  43.732265  15.893073
693577                               Ždiar (Tatra)  49.269550  20.266770
693579                        Žilina (bus station)  49.225414  18.747717
693581                                       Žireč  50.403313  15.847935
693583                         Županja bus station  45.077538  18.695953

[360388 rows x 3 columns]


In [ ]:
# 1. Remove rows where stop_status is "no data"
df = df.drop(columns=['trip_start_time', 'provider', 'trip_start_time'])
df = df[df["stop_status"] != "no data"]


# 2. Convert stop_status to 0/1 (NaN -> 0, "skipped" -> 1) and rename to "skipped"
df["stop_status"] = (df["stop_status"] == "skipped").astype(int)
df = df.rename(columns={"stop_status": "skipped"})



#Inspect rows with a blank stop_name
#print(df[df["stop_name"].isna()].to_string())

In [ ]:
# 3 Columns that must always have a value
cc = ["service_date","route","trip_id","stop_sequence","stop_id","stop_name","trip_status","skipped","delay_source","route_id"]

df3 = df.dropna(subset=cc)

both_empty = df3[["scheduled_arrival","scheduled_departure"]].isna().all(axis=1)
not_skipped = df3["skipped"] == 0
cancelled = df3["trip_status"] == "cancelled"
print(df3.shape[0])


21852159
0


In [18]:
merged_df = pd.merge(df3, stops, on='stop_name', how='left')
print(merged_df)
print(merged_df)

         service_date feed_id route  trip_id  stop_sequence  stop_id  \
0          2026-09-30     NaN  RB34  1000028              0   494459   
1          2026-09-30     NaN  RB34  1000028              1   442880   
2          2026-09-30     NaN  RB34  1000028              2   411746   
3          2026-09-30     NaN  RB34  1000028              3   555822   
4          2026-09-30     NaN  RB34  1000028              4   627959   
...               ...     ...   ...      ...            ...      ...   
21852154   2026-11-10     NaN  RS21   999965              4   406401   
21852155   2026-11-10     NaN  RS21   999965              5   497519   
21852156   2026-11-10     NaN  RS21   999965              6   258406   
21852157   2026-11-10     NaN  RS21   999965              7    37337   
21852158   2026-11-10     NaN  RS21   999965              8   661471   

                  stop_name    scheduled_arrival  scheduled_departure  \
0            Idar-Oberstein  2026-09-30 20:18:00  2026-09-30 2